In [1]:
# 🚀 Fresh Google Colab Setup 2025 (NumPy 2.0 Compatible)
!apt install -q -y libomp-dev

# Install latest versions that support NumPy 2.0
!pip install -q --upgrade \
    torch \
    transformers \
    sentence-transformers \
    faiss-cpu==1.12.0 \
    huggingface_hub \
    tqdm

print("✅ All packages installed!")

# Test everything
try:
    import numpy as np
    import torch
    import faiss
    import sentence_transformers
    import transformers

    print("🎉 SUCCESS! All packages working!")
    print(f"✅ NumPy: {np.__version__}")
    print(f"✅ PyTorch: {torch.__version__}")
    print(f"✅ FAISS: Available")
    print(f"✅ Sentence Transformers: {sentence_transformers.__version__}")
    print(f"✅ Transformers: {transformers.__version__}")
    print(f"✅ CUDA: {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
        print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

    # Quick FAISS test (IndexFlatIP over L2-normalized vectors)
    test_vectors = np.random.random((5, 128)).astype('float32')
    faiss.normalize_L2(test_vectors)
    index = faiss.IndexFlatIP(128)
    index.add(test_vectors)
    print(f"✅ FAISS: Working ({index.ntotal} vectors indexed, IP over normalized vectors)")

    print("\n🎯 Ready for embedding generation!")

except Exception as e:
    print(f"❌ Error: {e}")

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libomp-14-dev libomp5-14
Suggested packages:
  libomp-14-doc
The following NEW packages will be installed:
  libomp-14-dev libomp-dev libomp5-14
0 upgraded, 3 newly installed, 0 to remove and 35 not upgraded.
Need to get 738 kB of archives.
After this operation, 8,991 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 libomp5-14 amd64 1:14.0.0-1ubuntu1.1 [389 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 libomp-14-dev amd64 1:14.0.0-1ubuntu1.1 [347 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libomp-dev amd64 1:14.0-55~exp2 [3,074 B]
Fetched 738 kB in 1s (576 kB/s)
Selecting previously unselected package libomp5-14:amd64.
(Reading database ... 126435 files and directories currently installed.)
Preparing to unpack .../libomp5-14_1%3a14.0.0-1ubuntu1.1

In [2]:
import os
import torch

# Create expected project structure
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/processed/embeddings", exist_ok=True)

# Enhanced GPU detection
print(f"🔥 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    BATCH_SIZE = 64  # Larger batch for GPU
else:
    print("⚠️ Using CPU - consider enabling GPU in Runtime → Change runtime type → GPU")
    BATCH_SIZE = 16  # Smaller batch for CPU

# Upload ramayana.json
from google.colab import files
print("📥 Upload your ramayana.json file:")
uploaded = files.upload()

# Move to expected location
for fname in uploaded.keys():
    if fname.endswith(".json") and "ramayana" in fname.lower():
        os.replace(fname, "data/raw/ramayana.json")
        print(f"✅ JSON moved to data/raw/ramayana.json")

assert os.path.exists("data/raw/ramayana.json"), "❌ ramayana.json not found!"

# Configuration
JSON_PATH = "data/raw/ramayana.json"
OUT_DIR = "data/processed/embeddings"

# Model order (Primary → Secondary → Tertiary)
PRIMARY_MODEL = "BAAI/bge-m3"
FALLBACK_MODEL = "intfloat/multilingual-e5-large"  # you can switch to instruct variant if desired
FINAL_FALLBACK = "l3cube-pune/indic-sentence-similarity-sbert"

# Windowing parameters (1-unit overlap)
WINDOW_SIZE = 5     # atomic units per window
STRIDE = 4          # advance by 4 atomic units (1-unit overlap)
MIN_TAIL = 1        # minimum tail window units at end (ensures very short sargas are covered)

print(f"🎯 Configuration set - Batch size: {BATCH_SIZE}, Window: W={WINDOW_SIZE}, S={STRIDE}, min={MIN_TAIL}")

🔥 CUDA available: True
🎮 GPU: Tesla T4
💾 GPU Memory: 14.7 GB
📥 Upload your ramayana.json file:


Saving ramayana.json to ramayana.json
✅ JSON moved to data/raw/ramayana.json
🎯 Configuration set - Batch size: 64, Window: W=5, S=4, min=1


In [3]:
import os
import json
import time
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
import logging
import torch

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("colab-emb-gen-windowed")

def _resolve_max_len(model_id: str, default_len: int = 512) -> int:
    """
    Resolve max sequence length using the tokenizer (cap to 8192 for safety).
    """
    try:
        tok = AutoTokenizer.from_pretrained(model_id)
        ml = getattr(tok, "model_max_length", default_len)
        if isinstance(ml, int) and ml > 0 and ml < 100000:
            return min(8192, ml)
        return default_len
    except Exception:
        return default_len

def load_embedding_model(primary_id: str, fallback_id: str, final_fallback_id: str):
    """Load SentenceTransformer with robust fallback and tokenizer-aware max length."""
    # Try primary
    try:
        logger.info(f"🤖 Loading primary model: {primary_id}")
        model = SentenceTransformer(primary_id)
        model.max_seq_length = _resolve_max_len(primary_id, 512)
        logger.info(f"✅ Primary model loaded (max_seq_length={model.max_seq_length})")
        return model, primary_id
    except Exception as e:
        logger.warning(f"⚠️ Primary model failed: {e}")

    # Try fallback
    try:
        logger.info(f"🤖 Loading fallback model: {fallback_id}")
        model = SentenceTransformer(fallback_id)
        model.max_seq_length = _resolve_max_len(fallback_id, 512)
        logger.info(f"✅ Fallback model loaded (max_seq_length={model.max_seq_length})")
        return model, fallback_id
    except Exception as e:
        logger.warning(f"⚠️ Fallback model failed: {e}")

    # Final fallback
    logger.info(f"🤖 Loading final fallback model: {final_fallback_id}")
    model = SentenceTransformer(final_fallback_id)
    model.max_seq_length = _resolve_max_len(final_fallback_id, 512)
    logger.info(f"✅ Final fallback model loaded (max_seq_length={model.max_seq_length})")
    return model, final_fallback_id

def join_titles(titles):
    try:
        return " | ".join([t.strip() for t in (titles or []) if isinstance(t, str) and t.strip()])
    except Exception:
        return ""

def unit_display_label(kanda_num: int, sarga_num: int, verse_indices: list[int]) -> str:
    if not verse_indices:
        return f"{kanda_num}.{sarga_num}.?"
    start = verse_indices[0]
    end = verse_indices[-1]
    if start == end:
        return f"{kanda_num}.{sarga_num}.{start}"
    return f"{kanda_num}.{sarga_num}.{start}–{end}"

def build_windowed_corpus_from_json(json_path: str,
                                   window_size_units: int,
                                   stride_units: int,
                                   min_window_size_units: int,
                                   model_id_for_tokenizer: str):
    """
    Build a windowed corpus over atomic verse units (multi-verse blocks respected),
    composing EN + IAST text and preserving Kanda/Sarga/Titles + verse labels per line.
    """
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Prepare tokenizer diagnostics
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id_for_tokenizer)
        max_len = _resolve_max_len(model_id_for_tokenizer, 512)
    except Exception:
        tokenizer = None
        max_len = 512

    texts = []
    mappings = []

    total_windows = 0
    truncated_count = 0

    for kanda_obj in data:
        kanda_name = kanda_obj.get("kanda", "Unknown Kanda")
        sargas = kanda_obj.get("sargas", [])

        for sarga_obj in sargas:
            sarga_num = int(sarga_obj.get("sarga_index", 0))
            titles = sarga_obj.get("titles", []) or []
            joined_titles = join_titles(titles)
            verses = sarga_obj.get("verses", []) or []

            # Build atomic units (each item in verses is an atomic multi-verse unit)
            units = []
            for v in verses:
                verse_indices = v.get("verse_indices", []) or []
                verse_ids = v.get("verse_ids", []) or []
                en = (v.get("translation_en") or "").strip()
                iast = (v.get("shloka_iast") or "").strip()
                if not verse_ids:
                    continue
                # Derive kanda_num from verse_ids (e.g., "1-1-19")
                try:
                    k_num = int(str(verse_ids[0]).split("-")[0])
                except Exception:
                    k_num = 0

                units.append({
                    "kanda_num": k_num,
                    "sarga_num": sarga_num,
                    "kanda_name": kanda_name,
                    "verse_indices": verse_indices,
                    "verse_ids": verse_ids,
                    "en": en,
                    "iast": iast,
                    "titles_joined": joined_titles,
                })

            if not units:
                continue

            # Sliding window over units within this sarga (matches app logic)
            W = max(1, int(window_size_units))
            S = max(1, int(stride_units))
            MINW = max(1, int(min_window_size_units))
            n = len(units)
            i = 0
            last_end = 0

            while i + W <= n:
                window = units[i:i+W]

                # Compute start/end verse indices for display
                start_idx = window[0]["verse_indices"][0] if window[0]["verse_indices"] else None
                end_idx = window[-1]["verse_indices"][-1] if window[-1]["verse_indices"] else None

                # English section
                english_lines = [f"English: Verse {start_idx} to Verse {end_idx} :"]
                for u in window:
                    label = unit_display_label(u["kanda_num"], u["sarga_num"], u["verse_indices"]) \
                             if u["verse_indices"] else f"{u['kanda_num']}.{u['sarga_num']}.?"
                    english_lines.append(f"{label} {u['en']}")

                # IAST section
                iast_lines = [f"IAST: Verse {start_idx} to Verse {end_idx} :"]
                for u in window:
                    label = unit_display_label(u["kanda_num"], u["sarga_num"], u["verse_indices"]) \
                             if u["verse_indices"] else f"{u['kanda_num']}.{u['sarga_num']}.?"
                    iast_lines.append(f"{label} {u['iast']}")

                chunk = (
                    f"Kanda: {kanda_name}\n"
                    f"Sarga: {sarga_num}\n"
                    f"Titles: {joined_titles}\n\n"
                    + "\n".join(english_lines) + "\n\n"
                    + "\n".join(iast_lines)
                ).strip()

                text_for_embed = f"passage: {chunk}" if "e5" in (model_id_for_tokenizer or "").lower() else chunk
                texts.append(text_for_embed)

                # Build mapping entry
                flat_ids = []
                for u in window:
                    flat_ids.extend(list(u.get("verse_ids", [])))

                # Central anchor verse_id (mid unit, mid verse) for representativeness
                try:
                    mid_unit = window[len(window)//2]
                    vid_list = list(mid_unit.get("verse_ids", []))
                    anchor_id = vid_list[len(vid_list)//2] if vid_list else window[0].get("verse_ids", [None])[0]
                except Exception:
                    anchor_id = window[0].get("verse_ids", [None])[0]

                # Diagnostics: token count and potential truncation
                approx_token_count = None
                maybe_truncated = False
                if tokenizer is not None:
                    try:
                        ids = tokenizer(text_for_embed, add_special_tokens=True, truncation=False)["input_ids"]
                        approx_token_count = len(ids)
                        maybe_truncated = approx_token_count > max_len
                    except Exception:
                        pass

                if maybe_truncated:
                    truncated_count += 1

                mappings.append({
                    "kanda_num": window[0]["kanda_num"],
                    "sarga_num": window[0]["sarga_num"],
                    "start_verse_index": start_idx,
                    "end_verse_index": end_idx,
                    "verse_ids": flat_ids,
                    "anchor_verse_id": anchor_id,
                    "kanda_name": kanda_name,
                    "titles": titles,  # keep titles array for downstream enrichment
                    "approx_token_count": approx_token_count,
                    "model_max_seq_length": max_len,
                    "maybe_truncated": bool(maybe_truncated),
                })

                last_end = i + W
                i += S

            # Tail window handling:
            # - If stride==4 and n>=W, ensure one end-aligned window to cover the last units when needed.
            # - If n < W, include a small window if it meets MINW.
            if last_end < n:
                if n >= W:
                    tail_start = n - W
                    window = units[tail_start:n]
                else:
                    if n < MINW:
                        window = None
                    else:
                        window = units[0:n]

                if window:
                    start_idx = window[0]["verse_indices"][0] if window[0]["verse_indices"] else None
                    end_idx = window[-1]["verse_indices"][-1] if window[-1]["verse_indices"] else None

                    english_lines = [f"English: Verse {start_idx} to Verse {end_idx} :"]
                    for u in window:
                        label = unit_display_label(u["kanda_num"], u["sarga_num"], u["verse_indices"]) \
                                 if u["verse_indices"] else f"{u['kanda_num']}.{u['sarga_num']}.?"
                        english_lines.append(f"{label} {u['en']}")

                    iast_lines = [f"IAST: Verse {start_idx} to Verse {end_idx} :"]
                    for u in window:
                        label = unit_display_label(u["kanda_num"], u["sarga_num"], u["verse_indices"]) \
                                 if u["verse_indices"] else f"{u['kanda_num']}.{u['sarga_num']}.?"
                        iast_lines.append(f"{label} {u['iast']}")

                    chunk = (
                        f"Kanda: {kanda_name}\n"
                        f"Sarga: {sarga_num}\n"
                        f"Titles: {joined_titles}\n\n"
                        + "\n".join(english_lines) + "\n\n"
                        + "\n".join(iast_lines)
                    ).strip()

                    text_for_embed = f"passage: {chunk}" if "e5" in (model_id_for_tokenizer or "").lower() else chunk
                    texts.append(text_for_embed)

                    flat_ids = []
                    for u in window:
                        flat_ids.extend(list(u.get("verse_ids", [])))

                    try:
                        mid_unit = window[len(window)//2]
                        vid_list = list(mid_unit.get("verse_ids", []))
                        anchor_id = vid_list[len(vid_list)//2] if vid_list else window[0].get("verse_ids", [None])[0]
                    except Exception:
                        anchor_id = window[0].get("verse_ids", [None])[0]

                    approx_token_count = None
                    maybe_truncated = False
                    if tokenizer is not None:
                        try:
                            ids = tokenizer(text_for_embed, add_special_tokens=True, truncation=False)["input_ids"]
                            approx_token_count = len(ids)
                            maybe_truncated = approx_token_count > max_len
                        except Exception:
                            pass

                    if maybe_truncated:
                        truncated_count += 1

                    mappings.append({
                        "kanda_num": window[0]["kanda_num"],
                        "sarga_num": window[0]["sarga_num"],
                        "start_verse_index": start_idx,
                        "end_verse_index": end_idx,
                        "verse_ids": flat_ids,
                        "anchor_verse_id": anchor_id,
                        "kanda_name": kanda_name,
                        "titles": titles,
                        "approx_token_count": approx_token_count,
                        "model_max_seq_length": max_len,
                        "maybe_truncated": bool(maybe_truncated),
                    })

    logger.info(f"✅ Windowed corpus built from JSON: {len(texts)} passages (W={window_size_units}, S={stride_units})")
    logger.info(f"   Truncated windows (estimated): {truncated_count}")
    return texts, mappings

def generate_embeddings_windowed(json_path: str,
                                 out_dir: str,
                                 batch_size: int,
                                 primary_model: str,
                                 fallback_model: str,
                                 final_fallback: str,
                                 window_size_units: int,
                                 stride_units: int,
                                 min_window_size_units: int):
    """Generate windowed embeddings (English + IAST only) with mappings and metadata."""
    os.makedirs(out_dir, exist_ok=True)

    # Load model with fallback strategy
    model, model_id = load_embedding_model(primary_model, fallback_model, final_fallback)

    # Build windowed corpus
    texts, mappings = build_windowed_corpus_from_json(
        json_path=json_path,
        window_size_units=window_size_units,
        stride_units=stride_units,
        min_window_size_units=min_window_size_units,
        model_id_for_tokenizer=model_id
    )

    # Enhanced batch encoding with detailed progress
    logger.info(f"🧠 Generating embeddings with {model_id}...")
    logger.info(f"📊 Processing {len(texts)} texts in batches of {batch_size}")
    logger.info(f"🔧 Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

    start_time = time.time()
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="🧠 Embedding batches"):
        batch = texts[i:i+batch_size]
        batch_embeddings = model.encode(
            batch,
            normalize_embeddings=True,
            show_progress_bar=False,
            device='cuda' if torch.cuda.is_available() else 'cpu'
        )
        all_embeddings.append(batch_embeddings)

    embeddings = np.vstack(all_embeddings).astype(np.float32)
    total_time = time.time() - start_time

    # Metadata aligned with app
    metadata = {
        "model_id": model_id,
        "dim": int(embeddings.shape[1]),
        "count": int(embeddings.shape[0]),
        "normalization": "L2",
        "creation_time": time.strftime('%Y-%m-%d %H:%M:%S UTC'),
        "processing_time_seconds": round(total_time, 2),
        "processing_rate_texts_per_sec": round(len(texts) / max(total_time, 1e-6), 2),
        "batch_size": batch_size,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "languages_included": ["en", "iast"],
        "windowing": {
            "window_size_units": int(window_size_units),
            "stride_units": int(stride_units),
            "min_window_size_units": int(min_window_size_units),
            "unit_type": "atomic_verse",
            "multi_verse_policy": "atomic",
        },
        "source": os.path.basename(json_path),
    }

    # Save artifacts
    np.save(f"{out_dir}/embeddings.npy", embeddings)
    with open(f"{out_dir}/mappings.json", "w", encoding="utf-8") as f:
        json.dump(mappings, f, ensure_ascii=False, indent=2)
    with open(f"{out_dir}/metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    logger.info(f"✅ Embeddings saved to {out_dir}")
    logger.info(f"📊 Final shape: {embeddings.shape}")
    logger.info(f"⏱️  Total time: {total_time:.1f}s ({metadata['processing_rate_texts_per_sec']:.1f} texts/sec)")
    logger.info(f"🎯 Model used: {model_id}")

    return embeddings, mappings, metadata

print("✅ Windowed embedding functions loaded")

✅ Windowed embedding functions loaded


In [4]:
# Generate windowed embeddings with enhanced tracking
print("🚀 Starting VANI V3 windowed embeddings generation...")

embeddings, mappings, metadata = generate_embeddings_windowed(
    JSON_PATH,
    OUT_DIR,
    BATCH_SIZE,
    PRIMARY_MODEL,
    FALLBACK_MODEL,
    FINAL_FALLBACK,
    WINDOW_SIZE,
    STRIDE,
    MIN_TAIL
)

print(f"🎉 Embeddings generation complete!")
print(f"📊 Generated {metadata['count']} embeddings with {metadata['dim']} dimensions")
print(f"⚡ Performance: {metadata['processing_rate_texts_per_sec']:.1f} texts/second")

🚀 Starting VANI V3 windowed embeddings generation...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

🧠 Embedding batches: 100%|██████████| 72/72 [09:54<00:00,  8.25s/it]

🎉 Embeddings generation complete!
📊 Generated 4607 embeddings with 1024 dimensions
⚡ Performance: 7.8 texts/second


In [5]:
import faiss
import os
import numpy as np

def build_faiss_index(out_dir: str):
    """Build FAISS index for fast similarity search (inner product over L2-normalized vectors)."""
    embeddings_path = os.path.join(out_dir, "embeddings.npy")
    assert os.path.exists(embeddings_path), "❌ embeddings.npy not found"

    logger.info("🔍 Building FAISS index...")
    E = np.load(embeddings_path).astype(np.float32)
    assert E.ndim == 2 and E.size > 0, f"Invalid embeddings shape: {E.shape}"

    dim = E.shape[1]

    # Normalize for cosine similarity via inner product
    faiss.normalize_L2(E)

    # Exact search index
    index = faiss.IndexFlatIP(dim)
    index.add(E)

    # Save index
    index_path = os.path.join(out_dir, "index.faiss")
    faiss.write_index(index, index_path)

    logger.info(f"✅ FAISS index created: {index_path}")
    logger.info(f"📊 Index contains {index.ntotal} vectors of dimension {dim}")
    return index_path

# Build the index
index_path = build_faiss_index(OUT_DIR)
print(f"🎯 FAISS index ready for fast similarity search!")

🎯 FAISS index ready for fast similarity search!


In [6]:
import json
import os
import numpy as np
import faiss

def verify_outputs(out_dir: str):
    """Comprehensive verification of all generated files, including window metadata fields."""
    required_files = ["metadata.json", "mappings.json", "embeddings.npy", "index.faiss"]

    logger.info("🔍 Verifying generated files...")

    # Check file existence and sizes
    for fname in required_files:
        fpath = f"{out_dir}/{fname}"
        if not os.path.exists(fpath):
            logger.error(f"❌ Missing file: {fname}")
            return False
        size = os.path.getsize(fpath) / (1024 * 1024)  # MB
        logger.info(f"✅ {fname}: {size:.1f} MB")

    # Verify embeddings integrity
    embeddings = np.load(f"{out_dir}/embeddings.npy")
    with open(f"{out_dir}/metadata.json") as f:
        metadata = json.load(f)
    with open(f"{out_dir}/mappings.json") as f:
        mappings = json.load(f)

    # Consistency checks
    expected_count = int(metadata["count"])
    actual_emb_count = int(embeddings.shape[0])
    actual_map_count = int(len(mappings))

    if expected_count == actual_emb_count == actual_map_count:
        logger.info(f"✅ Count consistency verified: {actual_emb_count}")
    else:
        logger.error(f"❌ Count mismatch - Meta: {expected_count}, Emb: {actual_emb_count}, Map: {actual_map_count}")
        return False

    # Verify FAISS index
    try:
        index = faiss.read_index(f"{out_dir}/index.faiss")
        if index.ntotal == expected_count:
            logger.info(f"✅ FAISS index verified: {index.ntotal} vectors")
        else:
            logger.error(f"❌ FAISS index count mismatch: {index.ntotal} vs {expected_count}")
            return False
    except Exception as e:
        logger.error(f"❌ FAISS index verification failed: {e}")
        return False

    # Embedding dimension check
    if embeddings.shape[1] >= 384:
        logger.info(f"✅ Embedding dimension looks good: {embeddings.shape[1]}")
    else:
        logger.warning(f"⚠️ Unusual embedding dimension: {embeddings.shape[1]}")

    # Normalization check (random sample norms ~1.0)
    sample_norms = np.linalg.norm(embeddings[: min(100, len(embeddings))], axis=1)
    if np.allclose(sample_norms, 1.0, atol=1e-6):
        logger.info("✅ Embeddings are properly normalized")
    else:
        logger.warning("⚠️ Embeddings may not be normalized")

    # Window metadata sanity checks (aligned with generator)
    required_map_keys = {"verse_ids", "anchor_verse_id", "start_verse_index", "end_verse_index", "kanda_name", "sarga_num", "titles"}
    sample = mappings[0] if mappings else {}
    missing = [k for k in required_map_keys if k not in sample]
    if missing:
        logger.error(f"❌ mappings.json is missing required keys: {missing}")
        return False
    if not isinstance(sample.get("verse_ids", []), list) or not sample.get("anchor_verse_id"):
        logger.error("❌ verse_ids must be a list and anchor_verse_id must be non-empty")
        return False

    # Metadata checks
    if "languages_included" in metadata and metadata["languages_included"] == ["en", "iast"]:
        logger.info("✅ languages_included present: ['en', 'iast']")
    else:
        logger.warning("⚠️ languages_included missing or unexpected in metadata.json")

    win = metadata.get("windowing", {})
    for key in ["window_size_units", "stride_units", "min_window_size_units"]:
        if key not in win:
            logger.error(f"❌ metadata.windowing missing key: {key}")
            return False

    logger.info("🎉 All verification checks passed!")
    return True

# Run comprehensive verification
if verify_outputs(OUT_DIR):
    print("✅ All files verified and ready for download!")

    # Print final summary
    with open(f"{OUT_DIR}/metadata.json") as f:
        meta = json.load(f)

    print(f"""
    🎊 VANI V3 Windowed Embeddings Generation Complete!

    📊 Summary:
    - Model: {meta['model_id']}
    - Vectors: {meta['count']:,}
    - Dimensions: {meta['dim']}
    - Device: {meta['device'].upper()}
    - Windowing: W={meta['windowing']['window_size_units']}, S={meta['windowing']['stride_units']}, min={meta['windowing']['min_window_size_units']}
    """)
else:
    print("❌ Verification failed - please check outputs before downloading")

✅ All files verified and ready for download!

    🎊 VANI V3 Windowed Embeddings Generation Complete!

    📊 Summary:
    - Model: BAAI/bge-m3
    - Vectors: 4,607
    - Dimensions: 1024
    - Device: CUDA
    - Windowing: W=5, S=4, min=1
    


In [7]:
# Download all generated files to your local machine
from google.colab import files
import os

print("📤 Downloading VANI V3 embeddings files...")

files_to_download = ["metadata.json", "mappings.json", "embeddings.npy", "index.faiss"]

for fname in files_to_download:
    fpath = f"{OUT_DIR}/{fname}"
    if os.path.exists(fpath):
        print(f"⬇️  Downloading {fname}...")
        files.download(fpath)
    else:
        print(f"❌ File not found: {fname}")

print("""
🎉 Download complete!

📁 Copy these files to your VANI project:
- Place all 4 files in: data/processed/embeddings/

🚀 Next steps:
1. Copy files to your VANI folder
2. Ensure .env has DATABASE_PATH and EMBEDDINGS_DIR set (defaults are fine)
3. Run: uvicorn app.main:app --host 127.0.0.1 --port 8000
4. Open: http://127.0.0.1:8000

Your complete RAG system is ready! 🙏✨
""")

📤 Downloading VANI V3 embeddings files...
⬇️  Downloading metadata.json...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Downloading mappings.json...


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

⬇️  Downloading index.faiss...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 Download complete!

📁 Copy these files to your VANI project:
- Place all 4 files in: data/processed/embeddings/

🚀 Next steps:
1. Copy files to your VANI folder
2. Ensure .env has DATABASE_PATH and EMBEDDINGS_DIR set (defaults are fine)
3. Run: uvicorn app.main:app --host 127.0.0.1 --port 8000
4. Open: http://127.0.0.1:8000

Your complete RAG system is ready! 🙏✨

